# Part 3: Data Analysis

This notebook performs the data analysis steps outlined in Part 3 of the Rearc Data Quest. It loads data from S3, performs calculations, and generates the required reports.

In [1]:
import pandas as pd
import boto3
import json
import os
import sys

In [2]:
# --- Configuration ---
# Set the S3 bucket name and AWS region.
# You can set these as environment variables or replace the default values here.
S3_BUCKET_NAME = os.environ.get("S3_BUCKET_NAME", "quest-872180502324-ap-south-1-an")
AWS_REGION = os.environ.get("AWS_REGION", "ap-south-1")

if not S3_BUCKET_NAME:
    print("Error: S3_BUCKET_NAME not set. Please set it before running.", file=sys.stderr)
else:
    print(f"Using S3 Bucket: {S3_BUCKET_NAME}")
    print(f"Using AWS Region: {AWS_REGION}")

Using S3 Bucket: quest-872180502324-ap-south-1-an
Using AWS Region: ap-south-1


## Data Loading

The following function loads the time-series and population datasets from S3, performs initial cleaning, and returns them as pandas DataFrames.

In [3]:
def load_data_from_s3(bucket_name: str, region_name: str):
    """
    Loads and cleans the data from S3 into pandas DataFrames.
    """
    # Create storage options to pass to pandas for S3 access.
    storage_options = {'client_kwargs': {'region_name': region_name}}
    # Construct S3 paths.
    ts_data_path = f"s3://{bucket_name}/pr.data.0.Current"
    pop_data_path = f"s3://{bucket_name}/population_data.json"

    try:
        print(f"Loading time-series data from {ts_data_path}...")
        df_pr = pd.read_csv(ts_data_path, sep='\t', storage_options=storage_options)
        df_pr.columns = df_pr.columns.str.strip()
        for col in df_pr.select_dtypes(include=['object', 'string']):
            df_pr[col] = df_pr[col].str.strip()
    except Exception as e:
        print(f"Error loading time-series data from {ts_data_path}: {e}", file=sys.stderr)
        print("Please check your AWS credentials and ensure the file exists in your bucket.", file=sys.stderr)
        return None, None

    try:
        print(f"Loading population data from {pop_data_path}...")
        s3_client = boto3.client('s3', region_name=region_name)
        response = s3_client.get_object(Bucket=bucket_name, Key='population_data.json')
        pop_json_data = json.loads(response['Body'].read().decode('utf-8'))

        df_pop = pd.json_normalize(pop_json_data['data'])
        df_pop.rename(columns={'Year': 'year', 'Population': 'population'}, inplace=True)
    except Exception as e:
        print(f"Error loading population data from {pop_data_path}: {e}", file=sys.stderr)
        print("Please check your AWS credentials and ensure the file exists in your bucket.", file=sys.stderr)
        return None, None

    # --- Perform type conversions and cleaning upfront ---
    df_pr['value'] = pd.to_numeric(df_pr['value'], errors='coerce')
    df_pr.dropna(subset=['value'], inplace=True)
    df_pr['year'] = df_pr['year'].astype(int)
    df_pop['year'] = df_pop['year'].astype(int)

    print("Data loaded and cleaned successfully.")
    return df_pr, df_pop

In [4]:
df_pr, df_pop = load_data_from_s3(S3_BUCKET_NAME, AWS_REGION)

# Display the first few rows of each dataframe to verify they loaded correctly
if df_pr is not None and df_pop is not None:
    print("Time Series Data Head:")
    display(df_pr.head())
    print("\nPopulation Data Head:")
    display(df_pop.head())

Loading time-series data from s3://quest-872180502324-ap-south-1-an/pr.data.0.Current...


d:\rearc_quest\venv\lib\site-packages\fsspec\registry.py:298: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


Loading population data from s3://quest-872180502324-ap-south-1-an/population_data.json...


d:\rearc_quest\venv\lib\site-packages\boto3\compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Data loaded and cleaned successfully.
Time Series Data Head:


,series_id,year,period,value,footnote_codes
0,PRS30006011,1995,Q01,2.6,NaN
1,PRS30006011,1995,Q02,2.1,NaN
2,PRS30006011,1995,Q03,0.9,NaN
3,PRS30006011,1995,Q04,0.1,NaN
4,PRS30006011,1995,Q05,1.4,NaN



Population Data Head:


,Nation ID,Nation,year,population
0,01000US,United States,2013,316128839.0
1,01000US,United States,2014,318857056.0
2,01000US,United States,2015,321418821.0
3,01000US,United States,2016,323127515.0
4,01000US,United States,2017,325719178.0


---
## Report 1: US Population Analysis (2013-2018)

Generate the mean and the standard deviation of the annual US population across the years [2013, 2018] inclusive.

In [5]:
if df_pop is not None:
    df_pop_filtered = df_pop[(df_pop['year'] >= 2013) & (df_pop['year'] <= 2018)]
    population_mean = df_pop_filtered['population'].mean()
    population_std = df_pop_filtered['population'].std()

    print(f"Mean Population: {population_mean:,.0f}")
    print(f"Standard Deviation of Population: {population_std:,.0f}")

Mean Population: 322,069,808
Standard Deviation of Population: 4,158,441


---
## Report 2: Best Year per Time Series

For every `series_id`, find the *best year*: the year with the max/largest sum of "value" for all quarters in that year. Generate a report with each series id, the best year for that series, and the summed value for that year.

In [6]:
if df_pr is not None:
    df_yearly_sum = df_pr.groupby(['series_id', 'year'])['value'].sum().reset_index()

    # Use idxmax to efficiently find the index of the max value for each group
    idx = df_yearly_sum.groupby('series_id')['value'].idxmax()
    df_best_year = df_yearly_sum.loc[idx]

    print("Best Year per Time Series Report:")
    display(df_best_year)

Best Year per Time Series Report:


,series_id,year,value
27,PRS30006011,2022,20.500
58,PRS30006012,2022,17.100
65,PRS30006013,1998,705.895
108,PRS30006021,2010,17.700
139,PRS30006022,2010,12.400
...,...,...,...
8459,PRS88003192,2002,282.800
8512,PRS88003193,2024,862.564
8541,PRS88003201,2022,38.900
8572,PRS88003202,2022,29.700


---
## Report 3: Joined Time Series and Population Data

Generate a report that will provide the `value` for `series_id = PRS30006032` and `period = Q01` and the `population` for that given year (if available in the population dataset).

In [7]:
if df_pr is not None and df_pop is not None:
    # Filter for the specific series and period. Use .copy() to avoid SettingWithCopyWarning.
    df_pr_q1 = df_pr[(df_pr['series_id'] == 'PRS30006032') & (df_pr['period'] == 'Q01')].copy()

    # Use a 'left' merge to keep all time-series records and add population where available.
    df_joined = pd.merge(df_pr_q1, df_pop, on='year', how='left')

    # Select and reorder columns for the final report
    df_joined_report = df_joined[['series_id', 'year', 'period', 'value', 'population']]

    # Format the population column for better readability in the output
    df_joined_report['population'] = df_joined_report['population'].apply(
        lambda x: f'{x:,.0f}' if pd.notna(x) else 'N/A'
    )

    print("Joined Report for series_id='PRS30006032' and period='Q01':")
    display(df_joined_report)

Joined Report for series_id='PRS30006032' and period='Q01':


C:\Users\jagru\AppData\Local\Temp\ipykernel_44224\2354932539.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_joined_report['population'] = df_joined_report['population'].apply(


,series_id,year,period,value,population
0,PRS30006032,1995,Q01,0.0,N/A
1,PRS30006032,1996,Q01,-4.2,N/A
2,PRS30006032,1997,Q01,2.8,N/A
3,PRS30006032,1998,Q01,0.9,N/A
4,PRS30006032,1999,Q01,-4.1,N/A
5,PRS30006032,2000,Q01,0.5,N/A
6,PRS30006032,2001,Q01,-6.3,N/A
7,PRS30006032,2002,Q01,-6.6,N/A
8,PRS30006032,2003,Q01,-5.7,N/A
9,PRS30006032,2004,Q01,2.0,N/A
